# AmSC → Personal Access Token (PAT) Bootstrap Notebook

This notebook gets an **AmSC Personal Access Token (PAT)** from **MyAmSC** and saves it locally for the AmSC notebooks
(e.g. [`compute-jobs-amsc.ipynb`](compute-jobs-amsc.ipynb)). Those notebooks reach facility IRI APIs **through RIG** (Resource Integration Gateway)
instead of calling each facility directly with a facility-specific token.

A PAT is a long-lived (30 days) AmSC token issued by the AmSC identity provider. It is bound to the **one project** that was
selected in MyAmSC when you generated it (`amsc_project_context` claim). RIG uses that project to map you to the facility
project and to pick your facility credentials. To work in another project, switch project in MyAmSC and generate a new PAT.

It will:
1. Open MyAmSC, where you log in, select a project, and generate a PAT
2. Read the PAT (hidden input, so it does not end up in the notebook output)
3. Decode it and check expiration, issuer, and project
4. Verify it with RIG and list the facilities RIG serves
5. Save it to `~/.amsc_token.json` as `AMSC_PAT`

`~/.amsc_token.json` is separate from `~/.iri_token.json` (facility-specific tokens from the login notebooks in [`iri/`](../iri/)), so both can coexist.

## Optional Environment Variables
Set these in your shell or `.env` file:
- `AMSC_ENV` — AmSC deployment (default: `staging`)
- `AMSC_MYAMSC_URL`, `AMSC_RIG_URL`, `AMSC_CREDENTIAL_VAULT_URL`, `AMSC_ISSUER` — override the deployment URLs
- `AMSC_PAT` — use this PAT instead of pasting one

## 1) Setup and Imports

In [12]:
import sys
!{sys.executable} -m pip install requests python-dotenv

In [13]:
import os
import base64
import getpass
import json
import time
import webbrowser
import datetime as dt
from pathlib import Path
from typing import Any, List

import requests
from dotenv import load_dotenv

## 2) Configuration

In [14]:
# Load environment variables from .env (if present)
load_dotenv()

# AmSC deployments. Each one has its own identity provider, so a PAT generated
# in one deployment's MyAmSC is not accepted by another deployment's RIG.
AMSC_ENVIRONMENTS = {
    "staging": {
        "myamsc_url": "https://my.amsc.energy.gov/",
        "rig_url": "https://rig.staging.american-science-cloud.org",
        "credential_vault_url": "https://rig-ui.staging.american-science-cloud.org",
        "issuer": "https://identity.amsc.ornl.gov/am/oauth2",
    },
}

AMSC_ENV = os.getenv("AMSC_ENV", "staging")
env_defaults = AMSC_ENVIRONMENTS.get(AMSC_ENV, {})

MYAMSC_URL = os.getenv("AMSC_MYAMSC_URL", env_defaults.get("myamsc_url", "")).rstrip("/")
RIG_URL = os.getenv("AMSC_RIG_URL", env_defaults.get("rig_url", "")).rstrip("/")
CREDENTIAL_VAULT_URL = os.getenv("AMSC_CREDENTIAL_VAULT_URL", env_defaults.get("credential_vault_url", "")).rstrip("/")
EXPECTED_ISSUER = os.getenv("AMSC_ISSUER", env_defaults.get("issuer", ""))

if not MYAMSC_URL or not RIG_URL:
    raise RuntimeError(f"Unknown AMSC_ENV={AMSC_ENV!r}. Set AMSC_MYAMSC_URL and AMSC_RIG_URL.")

TOKEN_FILE = Path("~/.amsc_token.json").expanduser()
TIMEOUT = 30

print("AmSC environment:", AMSC_ENV)
print("MyAmSC          :", MYAMSC_URL)
print("RIG             :", RIG_URL)

AmSC environment: staging
MyAmSC          : https://my.amsc.energy.gov
RIG             : https://rig.staging.american-science-cloud.org


## 3) Helpers

In [15]:
def decode_jwt_part(part: str) -> dict:
    part += "=" * (-len(part) % 4)
    return json.loads(base64.urlsafe_b64decode(part))


def decode_jwt(token: str):
    """Decode a JWT for inspection only. Kong and RIG verify the signature."""
    parts = token.split(".")
    if len(parts) != 3:
        raise ValueError("Not a JWT (expected header.payload.signature).")
    return decode_jwt_part(parts[0]), decode_jwt_part(parts[1])


def fmt_ts(epoch: Any) -> str:
    return dt.datetime.fromtimestamp(int(epoch), tz=dt.UTC).strftime("%Y-%m-%d %H:%M:%SZ")


def print_table(headers: List[str], rows: List[List[Any]]) -> None:
    widths = [len(h) for h in headers]
    for row in rows:
        for i, cell in enumerate(row):
            widths[i] = max(widths[i], len(str(cell)))
    fmt = " | ".join(f"{{:<{w}}}" for w in widths)
    sep = "-+-".join("-" * w for w in widths)

    print(fmt.format(*headers))
    print(sep)
    for row in rows:
        print(fmt.format(*[str(c) for c in row]))
    print()

## 4) Log in to MyAmSC, select a project, and generate a PAT

Run the next cell. It opens MyAmSC in your browser (or use the printed link). Then:

1. **Log in** with your AmSC identity (MFA is required).
2. **Select a project** in the left sidebar. The PAT is bound to this project; RIG only acts on this project for calls made with it.
3. Go to **Personal Access Tokens** and click **New Token**.
4. **Copy the token.** It is shown only once.

Manage or revoke your PATs from the same MyAmSC page.

In [16]:
print(f"Open MyAmSC and generate a PAT:\n\n    {MYAMSC_URL}\n")
try:
    webbrowser.open(MYAMSC_URL, new=2)
except Exception:
    pass  # Headless or remote Jupyter: use the link above.

Open MyAmSC and generate a PAT:

    https://my.amsc.energy.gov



## 5) Paste the PAT
Input is hidden, so the token is not stored in the notebook output.

In [17]:
PAT = os.getenv("AMSC_PAT", "").strip()
if PAT:
    print("Using AMSC_PAT from the environment.")
else:
    PAT = getpass.getpass("Paste your AmSC PAT: ").strip()

# Accept a copied "Bearer <token>" value as well.
if PAT.lower().startswith("bearer "):
    PAT = PAT[len("bearer "):].strip()

if not PAT:
    raise RuntimeError("No PAT provided.")
print(f"Read PAT ({len(PAT)} characters).")

Paste your AmSC PAT:  ········


Read PAT (1671 characters).


## 6) Decode and check the PAT

In [18]:
header, claims = decode_jwt(PAT)

project = claims.get("amsc_project_context")
exp = claims.get("exp")

print("=== PAT claims ===")
for key in ("iss", "sub", "aud", "amsc_project_context", "jti", "scope", "iat", "exp"):
    if key in claims:
        value = fmt_ts(claims[key]) if key in ("iat", "exp") else claims[key]
        print(f"{key:21}: {value}")

if not exp:
    raise RuntimeError("PAT has no exp claim.")
if exp <= time.time():
    raise RuntimeError(f"PAT expired at {fmt_ts(exp)}. Generate a new one in MyAmSC.")
print(f"\nPAT expires in {(exp - time.time()) / 86400:.1f} days")

if not project or project == "undefined":
    raise RuntimeError("PAT has no amsc_project_context. Select a project in MyAmSC, then generate the PAT again.")

if EXPECTED_ISSUER and claims.get("iss") != EXPECTED_ISSUER:
    print(f"WARNING: issuer {claims.get('iss')!r} is not the {AMSC_ENV} issuer ({EXPECTED_ISSUER}). RIG will likely reject this PAT.")

=== PAT claims ===
iss                  : https://identity.amsc.ornl.gov/am/oauth2
sub                  : b1a126d9-398c-40d9-97e8-4abb51a37ec5
aud                  : https://api.amsc.energy.gov
amsc_project_context : amsc-rig-administration
jti                  : qbrnR7bxOwj4BTWWfcDuLet0MjY
scope                : ['amsc:none']
iat                  : 2026-10-08 12:01:23Z
exp                  : 2026-11-07 12:01:23Z

PAT expires in 30.0 days


## 7) Verify the PAT with RIG
RIG's `/ready` endpoint requires a valid AmSC bearer token and lists the facilities RIG proxies. A `401` means the PAT was
rejected (expired, revoked, or issued by a different deployment).

- **Tier 1** facilities accept your AmSC identity directly.
- **Tier 2** (e.g. ALCF via Globus) and **Tier 3** (vaulted, e.g. NERSC and OLCF) also need a facility credential connected in
  the Credential Vault for your project. [`compute-jobs-amsc.ipynb`](compute-jobs-amsc.ipynb) checks access to each facility.

In [19]:
r = requests.get(f"{RIG_URL}/ready", headers={"Authorization": f"Bearer {PAT}", "Accept": "application/json"}, timeout=TIMEOUT)
if r.status_code == 401:
    raise RuntimeError(f"RIG rejected the PAT (401): {r.text[:300]}")
if not r.ok:
    raise RuntimeError(f"RIG /ready failed: {r.status_code} {r.text[:300]}")

facilities = r.json().get("facilities", [])
print(f"RIG accepted the PAT. Facilities available through {RIG_URL}:\n")
print_table(
    ["Facility", "Tier", "Name", "API version"],
    [[f.get("name", ""), f.get("tier", ""), f.get("display_name") or "", f.get("api_version") or ""] for f in facilities],
)
if CREDENTIAL_VAULT_URL:
    print(f"Credential Vault (Tier 2/3 facility credentials): {CREDENTIAL_VAULT_URL}")

RIG accepted the PAT. Facilities available through https://rig.staging.american-science-cloud.org:

Facility      | Tier | Name | API version
--------------+------+------+------------
nersc         | 3    |      | v1         
alcf          | 2    |      | v1         
pnnl          | 1    |      | v1         
esnet-east    | 1    |      | v2         
esnet-west    | 1    |      | v2         
olcf-open     | 3    |      | v2         
olcf-moderate | 3    |      | v2         

Credential Vault (Tier 2/3 facility credentials): https://rig-ui.staging.american-science-cloud.org


## 8) Save PAT for Other Notebooks

In [20]:
token_data = {
    "AMSC_PAT": PAT,
    "AMSC_ENV": AMSC_ENV,
    "AMSC_RIG_URL": RIG_URL,
    "AMSC_MYAMSC_URL": MYAMSC_URL,
    "AMSC_CREDENTIAL_VAULT_URL": CREDENTIAL_VAULT_URL,
    "amsc_project_context": project,
    "sub": claims.get("sub"),
    "jti": claims.get("jti"),
    "expires_at": exp,
}

TOKEN_FILE.parent.mkdir(parents=True, exist_ok=True)
# The PAT stays valid for weeks: keep the file readable by the owner only.
fd = os.open(TOKEN_FILE, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
with os.fdopen(fd, "w") as f:
    json.dump(token_data, f, indent=2)
os.chmod(TOKEN_FILE, 0o600)  # also tighten a file that already existed
print(f"PAT saved: {TOKEN_FILE} (project={project}, expires {fmt_ts(exp)})")

PAT saved: /Users/jbalcas/.amsc_token.json (project=amsc-rig-administration, expires 2026-11-07 12:01:23Z)


## 9) Facility access: what else each facility needs

A valid PAT gets you into RIG, but **not every facility accepts it on its own**. Depending on the facility, your project
also needs resources at the site, or a facility credential configured in **RIG-UI → Credential Vault** (with the same
project active as in the PAT).

| Facility | RIG tier | What you need |
|---|---|---|
| ESnet (`esnet-east`, `esnet-west`), PNNL (`pnnl`) | 1 | Nothing in RIG-UI. The PAT's project must be active and have resources requested at the site. |
| NERSC (`nersc`) | 3 | A Globus token ([`iri/login-globus.ipynb`](../iri/login-globus.ipynb)) that NERSC has authorized, saved in the Credential Vault. |
| ALCF (`alcf`) | 2 | Log in with your ALCF credentials from the Credential Vault (via Globus); RIG stores and refreshes the token. |
| ORNL / OLCF (`olcf-open`, `olcf-moderate`) | 3 | A token from the ORNL (OLCF S3M) website, entered in the Credential Vault. |

Step-by-step instructions with screenshots are in **[`amsc-facility-access.ipynb`](amsc-facility-access.ipynb)**. Its last step re-checks your access to
every facility with this PAT.